In [1]:
import json
from pathlib import Path
from imgnet.collections.store import IndexedDatasets
from imgnet.collections.source import ZenodoSource
from imgtools.nifti.crawl import Crawler
from rich import print
import pandas as pd

/home/gpudual/bhklab/declan/med-image_index/.pixi/envs/default/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# LNDb

Incomplete: LNDb stores .mhd not nifti so i'm not sure if it can be processed the same way. 

In [2]:
DATASET_NAME = "LNDb"

indexed_datasets = IndexedDatasets("indexed_datasets") # This will download the latest version of the indexed_datasets if you dont provide a path
print(indexed_datasets.collections[:10])



14:34:13 [info     ] Indexed datasets path: /home/gpudual/bhklab/declan/med-image_index/indexed_datasets [imgnet] call=store.__init__:71


[
    '4D-Lung',
    'ACRIN-6698',
    'ACRIN-Contralateral-Breast-MR',
    'ACRIN-FLT-Breast',
    'ACRIN-NSCLC-FDG-PET',
    'AMOS',
    'Adrenal-ACC-Ki67-Seg',
    'Advanced-MRI-Breast-Lesions',
    'Anti-PD-1_Lung',
    'B-mode-and-CEUS-Liver'
]

In [3]:
description = """
The Lung Nodule Database (LNDb) was developed as an external dataset complimentary to LIDCIDRI. The publication of this database gives continuity to LIDC-IDRI and allows the community to perform an external and comparable validation of proposed computer-aided diagnosis (CAD) systems.

The LNDb contains 294 CT scans collected retrospectively at the Centro Hospitalar e Universitário de São João (CHUSJ) in Porto, Portugal between 2016 and 2018. All data was acquired under approval from the CHUSJ Ethical Commitee and was anonymised prior to any analysis to remove personal information except for patient birth year and gender.

The database served as the basis for the Grand Challenge on automatic lung cancer patient management, or LNDb challenge.
"""

In [4]:
source = ZenodoSource(
    file_type="nifti",
    source="zenodo",
    record_id="8348419",
    description=description,
    #post_download=["unzip"]
)

In [5]:
dataset_index_path = (indexed_datasets.path / DATASET_NAME)
dataset_index_path.mkdir(parents=True, exist_ok=True)
(dataset_index_path / "parquet").mkdir(exist_ok=True)
with open(dataset_index_path / "source.json", "w") as f:
    json.dump(source.model_dump(mode="json"), f, indent=2)

In [6]:

print(indexed_datasets.source_config(DATASET_NAME))

ZenodoSource(
    file_type=<FileType.NIFTI: 'nifti'>,
    source='zenodo',
    record_id='8348419',
    filenames=None,
    post_download=['unzip'],
    description='\nThe Lung Nodule Database (LNDb) was developed as an external dataset complimentary to LIDCIDRI. 
The publication of this database gives continuity to LIDC-IDRI and allows the community to perform an external and 
comparable validation of proposed computer-aided diagnosis (CAD) systems.\n\nThe LNDb contains 294 CT scans 
collected retrospectively at the Centro Hospitalar e Universitário de São João (CHUSJ) in Porto, Portugal between 
2016 and 2018. All data was acquired under approval from the CHUSJ Ethical Commitee and was anonymised prior to any
analysis to remove personal information except for patient birth year and gender.\n\nThe database served as the 
basis for the Grand Challenge on automatic lung cancer patient management, or LNDb challenge.\n'
)

In [7]:
temp_data_path = Path("temp_data") / DATASET_NAME
temp_data_path.mkdir(parents=True, exist_ok=True)
#downloader = indexed_datasets.downloader(DATASET_NAME).download(temp_data_path)

In [8]:
import os
import rarfile
from pathlib import Path

def extract_all_rar_files_rarfile(folder_path):
    """
    Extract all .rar files using rarfile library (requires unrar).
    
    Args:
        folder_path (str): Path to the folder containing .rar files
    """
    folder = Path(folder_path)
    
    if not folder.exists():
        raise ValueError(f"Folder does not exist: {folder_path}")
    
    rar_files = list(folder.glob("*.rar"))
    
    if not rar_files:
        print(f"No .rar files found in {folder_path}")
        return
    
    for rar_file in rar_files:
        try:
            print(f"Extracting: {rar_file.name}")
            
            with rarfile.RarFile(str(rar_file)) as rf:
                rf.extractall(path=str(folder))
            
            print(f"Successfully extracted: {rar_file.name}")
            
        except Exception as e:
            print(f"Error extracting {rar_file.name}: {e}")

# Usage
extract_all_rar_files_rarfile(temp_data_path)

Extracting: data0.rar

Successfully extracted: data0.rar

Extracting: data1.rar

Successfully extracted: data1.rar

Extracting: masks.rar

Successfully extracted: masks.rar

... (8 outputs left out)

Extracting: data3.rar

Successfully extracted: data3.rar

Extracting: testdata0.rar

Successfully extracted: testdata0.rar

In [9]:
crawler = Crawler(
    nifti_dir=temp_data_path,
    scan_name_pattern="COVID-19-CT-Seg_20cases/{_throwaway}_{ImageID}.nii.gz",
    mask_name_pattern="{ROINames}_Mask/{_throwaway}_{ImageID}.nii.gz",
    output_dir=indexed_datasets.path,
    n_jobs=10,
    force=True,
    deep=False,
)
crawler.crawl()


FileNotFoundError: No matching files in /home/gpudual/bhklab/declan/med-image_index/temp_data/LNDb

In [ ]:
index_df = pd.read_csv(indexed_datasets.path / "COVID-19 CT Lung/index.csv")

index_df



,_throwaway,ImageID,ROINames,filepath,file_type,reference_id,reference_scan
0,coronacases,1,NaN,COVID-19-CT-Seg_20cases/coronacases_001.nii.gz,scan,coronacases_001,NaN
1,coronacases,2,NaN,COVID-19-CT-Seg_20cases/coronacases_002.nii.gz,scan,coronacases_002,NaN
2,coronacases,3,NaN,COVID-19-CT-Seg_20cases/coronacases_003.nii.gz,scan,coronacases_003,NaN
3,coronacases,4,NaN,COVID-19-CT-Seg_20cases/coronacases_004.nii.gz,scan,coronacases_004,NaN
4,coronacases,5,NaN,COVID-19-CT-Seg_20cases/coronacases_005.nii.gz,scan,coronacases_005,NaN
...,...,...,...,...,...,...,...
75,radiopaedia_29_86491,1,Lung_and_Infection,Lung_and_Infection_Mask/radiopaedia_29_86491_1...,mask,radiopaedia_29_86491_1,COVID-19-CT-Seg_20cases/radiopaedia_29_86491_1...
76,radiopaedia_36_86526,0,Lung_and_Infection,Lung_and_Infection_Mask/radiopaedia_36_86526_0...,mask,radiopaedia_36_86526_0,COVID-19-CT-Seg_20cases/radiopaedia_36_86526_0...
77,radiopaedia_40_86625,0,Lung_and_Infection,Lung_and_Infection_Mask/radiopaedia_40_86625_0...,mask,radiopaedia_40_86625_0,COVID-19-CT-Seg_20cases/radiopaedia_40_86625_0...
78,radiopaedia_4_85506,1,Lung_and_Infection,Lung_and_Infection_Mask/radiopaedia_4_85506_1....,mask,radiopaedia_4_85506_1,COVID-19-CT-Seg_20cases/radiopaedia_4_85506_1....


In [ ]:
index_df

,_throwaway,ImageID,ROINames,filepath,file_type,reference_id,reference_scan
0,coronacases,1,NaN,COVID-19-CT-Seg_20cases/coronacases_001.nii.gz,scan,coronacases_001,NaN
1,coronacases,2,NaN,COVID-19-CT-Seg_20cases/coronacases_002.nii.gz,scan,coronacases_002,NaN
2,coronacases,3,NaN,COVID-19-CT-Seg_20cases/coronacases_003.nii.gz,scan,coronacases_003,NaN
3,coronacases,4,NaN,COVID-19-CT-Seg_20cases/coronacases_004.nii.gz,scan,coronacases_004,NaN
4,coronacases,5,NaN,COVID-19-CT-Seg_20cases/coronacases_005.nii.gz,scan,coronacases_005,NaN
...,...,...,...,...,...,...,...
75,radiopaedia_29_86491,1,Lung_and_Infection,Lung_and_Infection_Mask/radiopaedia_29_86491_1...,mask,radiopaedia_29_86491_1,COVID-19-CT-Seg_20cases/radiopaedia_29_86491_1...
76,radiopaedia_36_86526,0,Lung_and_Infection,Lung_and_Infection_Mask/radiopaedia_36_86526_0...,mask,radiopaedia_36_86526_0,COVID-19-CT-Seg_20cases/radiopaedia_36_86526_0...
77,radiopaedia_40_86625,0,Lung_and_Infection,Lung_and_Infection_Mask/radiopaedia_40_86625_0...,mask,radiopaedia_40_86625_0,COVID-19-CT-Seg_20cases/radiopaedia_40_86625_0...
78,radiopaedia_4_85506,1,Lung_and_Infection,Lung_and_Infection_Mask/radiopaedia_4_85506_1....,mask,radiopaedia_4_85506_1,COVID-19-CT-Seg_20cases/radiopaedia_4_85506_1....


In [ ]:
def map_modality(row):
    if row["file_type"] == "scan":
            return "CT"
    else:
        return "SEG"
    

index_df["Modality"] = index_df.apply(map_modality, axis=1)
index_df["BodyPartExamined"] = "LUNGS"


index_df

,_throwaway,ImageID,ROINames,filepath,file_type,reference_id,reference_scan,Modality,BodyPartExamined
0,coronacases,1,NaN,COVID-19-CT-Seg_20cases/coronacases_001.nii.gz,scan,coronacases_001,NaN,CT,LUNGS
1,coronacases,2,NaN,COVID-19-CT-Seg_20cases/coronacases_002.nii.gz,scan,coronacases_002,NaN,CT,LUNGS
2,coronacases,3,NaN,COVID-19-CT-Seg_20cases/coronacases_003.nii.gz,scan,coronacases_003,NaN,CT,LUNGS
3,coronacases,4,NaN,COVID-19-CT-Seg_20cases/coronacases_004.nii.gz,scan,coronacases_004,NaN,CT,LUNGS
4,coronacases,5,NaN,COVID-19-CT-Seg_20cases/coronacases_005.nii.gz,scan,coronacases_005,NaN,CT,LUNGS
...,...,...,...,...,...,...,...,...,...
75,radiopaedia_29_86491,1,Lung_and_Infection,Lung_and_Infection_Mask/radiopaedia_29_86491_1...,mask,radiopaedia_29_86491_1,COVID-19-CT-Seg_20cases/radiopaedia_29_86491_1...,SEG,LUNGS
76,radiopaedia_36_86526,0,Lung_and_Infection,Lung_and_Infection_Mask/radiopaedia_36_86526_0...,mask,radiopaedia_36_86526_0,COVID-19-CT-Seg_20cases/radiopaedia_36_86526_0...,SEG,LUNGS
77,radiopaedia_40_86625,0,Lung_and_Infection,Lung_and_Infection_Mask/radiopaedia_40_86625_0...,mask,radiopaedia_40_86625_0,COVID-19-CT-Seg_20cases/radiopaedia_40_86625_0...,SEG,LUNGS
78,radiopaedia_4_85506,1,Lung_and_Infection,Lung_and_Infection_Mask/radiopaedia_4_85506_1....,mask,radiopaedia_4_85506_1,COVID-19-CT-Seg_20cases/radiopaedia_4_85506_1....,SEG,LUNGS


In [ ]:
from imgnet.collections.store import IndexedDatasets
from imgindex.model import validate_index
from imgindex.parquet import csv_to_parquet


index_df.rename(columns={"reference_id": "SampleID"}, inplace=True)
index_df.to_csv(indexed_datasets.path / DATASET_NAME / "index.csv", index=False)
csv_to_parquet(index_df, indexed_datasets.path / DATASET_NAME)
validate_index(index_df, "nifti", lazy=True)

,_throwaway,ImageID,ROINames,filepath,file_type,SampleID,reference_scan,Modality,BodyPartExamined
0,coronacases,1,NaN,COVID-19-CT-Seg_20cases/coronacases_001.nii.gz,scan,coronacases_001,NaN,CT,LUNGS
1,coronacases,2,NaN,COVID-19-CT-Seg_20cases/coronacases_002.nii.gz,scan,coronacases_002,NaN,CT,LUNGS
2,coronacases,3,NaN,COVID-19-CT-Seg_20cases/coronacases_003.nii.gz,scan,coronacases_003,NaN,CT,LUNGS
3,coronacases,4,NaN,COVID-19-CT-Seg_20cases/coronacases_004.nii.gz,scan,coronacases_004,NaN,CT,LUNGS
4,coronacases,5,NaN,COVID-19-CT-Seg_20cases/coronacases_005.nii.gz,scan,coronacases_005,NaN,CT,LUNGS
...,...,...,...,...,...,...,...,...,...
75,radiopaedia_29_86491,1,Lung_and_Infection,Lung_and_Infection_Mask/radiopaedia_29_86491_1...,mask,radiopaedia_29_86491_1,COVID-19-CT-Seg_20cases/radiopaedia_29_86491_1...,SEG,LUNGS
76,radiopaedia_36_86526,0,Lung_and_Infection,Lung_and_Infection_Mask/radiopaedia_36_86526_0...,mask,radiopaedia_36_86526_0,COVID-19-CT-Seg_20cases/radiopaedia_36_86526_0...,SEG,LUNGS
77,radiopaedia_40_86625,0,Lung_and_Infection,Lung_and_Infection_Mask/radiopaedia_40_86625_0...,mask,radiopaedia_40_86625_0,COVID-19-CT-Seg_20cases/radiopaedia_40_86625_0...,SEG,LUNGS
78,radiopaedia_4_85506,1,Lung_and_Infection,Lung_and_Infection_Mask/radiopaedia_4_85506_1....,mask,radiopaedia_4_85506_1,COVID-19-CT-Seg_20cases/radiopaedia_4_85506_1....,SEG,LUNGS
